# Agrarpolitik im Wandel? – Bundestag 2018–2025

Dieses Notebook führt von den ursprünglichen Plenarreden über die Korpusaufbereitung und Wörterbuchanwendung bis zur Excel-Auswertung und zur Berechnung der Validierungskennzahlen. Die verwendeten Programme und Dateien sind bei jedem Schritt verlinkt.

Die Vorbereitung und die vier Schritte werden der Reihe nach ausgeführt. Als Laufzeit genügt **Python 3 mit CPU**. **Verlinkte Ergebnisdateien zeigen den bereitgestellten Abgabestand. Neu berechnete Ergebnisse entstehen während dieses Durchlaufs und können anschließend heruntergeladen werden.**

## Vorbereitung: Projektdateien laden

Programme, Wörterbücher, Korrekturdaten, Datengrundlage und Validierungsdateien werden aus dem [GitHub-Abgabestand](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/tree/abgabe-v1.0/) geladen. Die zusätzlich benötigte Python-Bibliothek ist in [requirements.txt](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/requirements.txt) festgelegt. Für den automatischen Download müssen Repository und Tag `abgabe-v1.0` öffentlich erreichbar sein.


In [ ]:
#@title Projektdateien aus GitHub laden
# Diese Angaben bestimmen die gemeinsam verwendete Fassung aller Projektdateien.
REPOSITORY_URL = "https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025"
ABGABESTAND = "abgabe-v1.0"

# Python-Bibliotheken für Dateizugriff und Programmstarts laden.
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import uuid
from pathlib import Path

# Die Vorbereitung beginnt einen neuen Durchlauf. Ergebnisse einer vorherigen
# Ausführung werden zurückgesetzt, damit keine unterschiedlichen Läufe vermischt werden.
PROJEKT = QUELLE = WOERTERBUCH_ORDNER = KORPUS = SATZMASTER = XLSX = None
PRUEFUNG_MASTER = PRUEFUNG_EXCEL = GUETEMASSE = PRUEFUNG_VALIDIERUNG = None
ablauf = ARBEIT = None
DOWNLOADS = {}
HERKUNFT = {}


def ausfuehren(argumente, zeige_fortschritt=True):
    """Startet die angegebene Python-Datei mit ihren Eingabe- und Ausgabepfaden.

    Die Berechnung steht in der aufgerufenen Datei. Diese Funktion wartet auf
    deren Abschluss und bricht bei einem Fehler ab. Bei zeige_fortschritt=False
    erscheinen Meldungen nur im Fehlerfall.
    """
    ausgabe = []
    with subprocess.Popen([sys.executable, "-u", *map(str, argumente)],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as prozess:
        for zeile in prozess.stdout:
            ausgabe.append(zeile)
            if zeige_fortschritt:
                print(zeile, end="", flush=True)
        if prozess.wait() != 0:
            if not zeige_fortschritt:
                print("".join(ausgabe))
            raise RuntimeError("Der Schritt wurde abgebrochen. Die Meldung oberhalb nennt die Ursache.")


# Die Angaben vor dem Download prüfen und bei fehlendem Projektzugang abbrechen.
if not REPOSITORY_URL.strip():
    raise ValueError("Die endgültige GitHub-Repository-Adresse fehlt noch in REPOSITORY_URL.")
repository_url = REPOSITORY_URL.strip().rstrip("/")
if not re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", repository_url):
    raise ValueError("REPOSITORY_URL muss die HTTPS-Adresse des GitHub-Repositorys enthalten.")
if not shutil.which("git"):
    raise RuntimeError("Git ist nicht verfügbar. Dieses Notebook ist für Google Colab vorbereitet.")
# Ausschließlich den benannten Tag laden, niemals automatisch auf einen anderen Stand wechseln.
referenz = "refs/tags/" + ABGABESTAND
if not ABGABESTAND or subprocess.run(["git", "check-ref-format", referenz],
                                     capture_output=True).returncode != 0:
    raise ValueError("ABGABESTAND muss einen gültigen Git-Tag enthalten, beispielsweise abgabe-v1.0.")

# Den festgelegten Abgabestand in einen eigenen Ordner der Colab-Sitzung laden.
# Die ermittelte Commit-Kennung dokumentiert den tatsächlich geladenen Inhalt.
arbeit = Path(tempfile.mkdtemp(prefix="agrarpolitik_"))
projekt = arbeit / "projekt"
umgebung = dict(os.environ, GIT_TERMINAL_PROMPT="0")
try:
    subprocess.run(["git", "init", "--quiet", str(projekt)], check=True, env=umgebung)
    subprocess.run(["git", "-C", str(projekt), "remote", "add", "origin", repository_url],
                   check=True, env=umgebung)
    subprocess.run(["git", "-C", str(projekt), "fetch", "--quiet", "--depth", "1", "origin", referenz],
                   check=True, env=umgebung)
    subprocess.run(["git", "-C", str(projekt), "checkout", "--quiet", "--detach", "FETCH_HEAD"],
                   check=True, env=umgebung)
    commit = subprocess.check_output(["git", "-C", str(projekt), "rev-parse", "HEAD"],
                                     text=True, env=umgebung).strip()
except subprocess.CalledProcessError as exc:
    shutil.rmtree(arbeit, ignore_errors=True)
    raise RuntimeError("Der GitHub-Abgabestand konnte nicht geladen werden. Repository und Release-Tag müssen öffentlich erreichbar sein.") from exc

# Prüfen, ob die zentralen Programme, Wörterbücher und die Excel-Vorlage vorliegen.
for name in ["requirements.txt", "src/ablauf.py", "korpusaufbereitung/korpus_aufbereiten.py",
             "src/satzmatcher.py", "reproduktion/satzmaster_pruefen.py",
             "auswertung/tabellen_neu_berechnen.py", "auswertung/ooxml_engine.py",
             "ergebnisse/Datengrundlage.xlsx",
             "validierung/kennzahlen_berechnen.py",
             "validierung/Validierung_AE_und_Themen_1000.xlsx",
             "validierung/Validierung_AE_Recall_3000.xlsx",
             "dictionaries/Woerterbuch_Agrar_Ernaehrung.json",
             "dictionaries/Woerterbuch_Oekologie.json",
             "dictionaries/Woerterbuch_Oekonomie.json",
             "dictionaries/Woerterbuch_Sicherheit_und_Resilienz.json"]:
    if not (projekt / name).is_file():
        raise ValueError("Eine benötigte Datei fehlt im Abgabestand: " + name)

# Die Softwareabhängigkeiten stammen aus demselben Abgabestand wie die Programme.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(projekt / "requirements.txt")],
               check=True)
# Die Hilfsfunktionen laden; der Zenodo-Download beginnt in der nächsten Zelle.
spec = importlib.util.spec_from_file_location("ablauf_" + uuid.uuid4().hex,
                                             projekt / "src/ablauf.py")
hilfsmodul = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hilfsmodul)

# Erst nach vollständiger Vorbereitung werden die Dateien für die nächsten Zellen freigegeben.
PROJEKT, ARBEIT, ablauf = projekt, arbeit, hilfsmodul
WOERTERBUCH_ORDNER = PROJEKT / "dictionaries"
HERKUNFT = {"Repository": repository_url, "Abgabestand (Tag)": ABGABESTAND,
            "Programmstand (Commit)": commit, "Quelle (DOI)": "10.5281/zenodo.18177196"}
print("Projektdateien und vier Wörterbücher bereit.")
print("Abgabestand:", ABGABESTAND, "– Commit:", commit[:12])


### Ausgangsreden laden

Die ursprünglichen Reden stammen aus dem [CPP-BT-Datensatz auf Zenodo, Version 17.01.2026](https://zenodo.org/records/18177196). Das Redenarchiv `CPP-BT_2026-01-17_DE_CSV_Reden_Gesamt.zip` wird heruntergeladen und anhand seiner Größe und Prüfsumme kontrolliert.

**Verwendete Dateien:** [src/ablauf.py](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/src/ablauf.py), Funktion `quelle_herunterladen`.

**Ergebnis:** Die unveränderte Ausgangsquelle steht für die Aufbereitung bereit.


In [ ]:
#@title Ausgangsreden von Zenodo laden
if PROJEKT is None:
    raise RuntimeError("Zuerst die Projektdateien erfolgreich aus GitHub laden.")
# Abhängige Ergebnisse zurücksetzen und einen eigenen Ordner für die Quelle anlegen.
QUELLE = KORPUS = SATZMASTER = XLSX = None
PRUEFUNG_MASTER = PRUEFUNG_EXCEL = GUETEMASSE = PRUEFUNG_VALIDIERUNG = None
DOWNLOADS = {}
quelle_ordner = Path(tempfile.mkdtemp(prefix="quelle_", dir=ARBEIT))
quelle = ablauf.quelle_herunterladen(
    "https://zenodo.org/records/18177196/files/CPP-BT_2026-01-17_DE_CSV_Reden_Gesamt.zip?download=1",
    quelle_ordner / "CPP-BT_2026-01-17_DE_CSV_Reden_Gesamt.zip",
    erwartete_bytes=117_814_684,
    erwartete_md5="9b03325c65c6930bc5e44d4206ce3d42",
)
# Erst den erfolgreich geprüften Download an Schritt 1 weitergeben.
QUELLE = quelle
print("Ausgangsreden vollständig geladen und geprüft. Weiter mit Schritt 1.")

## 1. Korpus aufbereiten

Die Reden werden für den Zeitraum 2018–2025 aufbereitet. Festgelegte Ergänzungen und Korrekturen werden angewendet, die bereinigten Texte anschließend in Sätze aufgeteilt.

**Verwendete Dateien:**

- [korpus_aufbereiten.py](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/korpusaufbereitung/korpus_aufbereiten.py): führt die Aufbereitung und Satzsegmentierung durch.
- [konfiguration/](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/tree/abgabe-v1.0/korpusaufbereitung/konfiguration): enthält Ergänzungen, Korrekturen, Zuordnungen und Prüfwerte.

**Neu erzeugt:** `Saetze_2018-2025.csv.gz`. Diese vollständige Satzbasis wird an die Wörterbuchanwendung und die Auswertung weitergegeben. Fortschrittsmeldungen erscheinen unter der Ausführungszelle.

**Bereitgestellter Abgabestand:** [Satzbasis beim Release herunterladen](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/releases/download/abgabe-v1.0/Saetze_2018-2025.csv.gz). Für diesen Durchlauf ist der Download nicht erforderlich, da die Satzbasis neu erzeugt wird.


In [ ]:
#@title Korpusaufbereitung starten
if PROJEKT is None or QUELLE is None or WOERTERBUCH_ORDNER is None:
    raise RuntimeError("Zuerst die Projektdateien und die Ausgangsreden erfolgreich laden.")
# Einen neuen Ergebnisordner verwenden und abhängige Ergebnisse zurücksetzen.
KORPUS = SATZMASTER = XLSX = None
PRUEFUNG_MASTER = PRUEFUNG_EXCEL = GUETEMASSE = PRUEFUNG_VALIDIERUNG = None
DOWNLOADS = {}
lauf = Path(tempfile.mkdtemp(prefix="aufbereitung_", dir=ARBEIT))
korpus = lauf / "korpus"
ausfuehren([PROJEKT / "korpusaufbereitung/korpus_aufbereiten.py",
            "--quelle", QUELLE, "--ausgabe", korpus, "--prozesse", "2"])
# Nur eine erfolgreich geprüfte Satzbasis für die Wörterbuchanwendung freigeben.
bilanz = json.loads((korpus / "Laufbilanz.json").read_text(encoding="utf-8"))
if bilanz.get("status") != "alle_Referenzpruefungen_bestanden":
    raise RuntimeError("Die Aufbereitung wurde nicht erfolgreich geprüft.")
KORPUS = korpus
print("Aufbereitete Plenarsätze:", f"{bilanz['satzzeilen']:,}".replace(",", "."))
print("Die Satzbasis wird automatisch an Schritt 2 weitergegeben.")


## 2. Wörterbücher anwenden

Aus der Satzbasis werden zunächst die Agrar- und Ernährungssätze ausgewählt. Anschließend erfolgt die Zuordnung zu den drei Themenbereichen. Mehrfachzuordnungen sind möglich.

**Verwendete Dateien:**

- [src/satzmatcher.py](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/src/satzmatcher.py): steuert die Auswahl der A&E-Sätze und die anschließende Themenzuordnung.

| Funktion | Wörterbuch mit den Regeln | Programm zur Anwendung |
| --- | --- | --- |
| Politikfeld auswählen | [Agrar und Ernährung](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/dictionaries/Woerterbuch_Agrar_Ernaehrung.json) | [ae_matcher.py](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/src/ae_matcher.py) |
| Ökologische Themen zuordnen | [Ökologie](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/dictionaries/Woerterbuch_Oekologie.json) | [oekologie_matcher.py](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/src/oekologie_matcher.py) |
| Wirtschaftliche Themen zuordnen | [Ökonomie](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/dictionaries/Woerterbuch_Oekonomie.json) | [oekonomie_matcher.py](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/src/oekonomie_matcher.py) |
| Sicherheits- und resilienzbezogene Themen zuordnen | [Sicherheit und Resilienz](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/dictionaries/Woerterbuch_Sicherheit_und_Resilienz.json) | [sr_matcher.py](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/src/sr_matcher.py) |

Das [Codebuch](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/codebuch/Codebuch.pdf) erläutert die Wörterbücher. Das Hilfsmodul [source_spans.py](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/src/source_spans.py) ordnet A&E-Treffer ihren Fundstellen im Originaltext zu.

**Technische Kontrolle:** [satzmaster_pruefen.py](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/reproduktion/satzmaster_pruefen.py) vergleicht den erzeugten Satzmaster mit den festgehaltenen Vergleichswerten in [referenz_satzmaster.json](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/reproduktion/referenz_satzmaster.json).

**Neu erzeugt:** `Satzmaster_Agrar_Ernaehrung.csv.gz` mit Satztexten, Metadaten, Themenzuordnungen und Trefferbelegen. Er bildet die Grundlage der anschließenden Auswertung.

**Bereitgestellter Abgabestand:** [A&E-Satzmaster](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/ergebnisse/Satzmaster_Agrar_Ernaehrung.csv.gz).


In [ ]:
#@title Wörterbuchanwendung starten
if KORPUS is None or WOERTERBUCH_ORDNER is None:
    raise RuntimeError("Zuerst Schritt 1 erfolgreich abschließen.")
# Für diesen Wörterbuchlauf neue Ausgaben anlegen und Folgeergebnisse zurücksetzen.
SATZMASTER = XLSX = PRUEFUNG_MASTER = PRUEFUNG_EXCEL = GUETEMASSE = PRUEFUNG_VALIDIERUNG = None
DOWNLOADS = {}
analyse = Path(tempfile.mkdtemp(prefix="woerterbuchlauf_", dir=ARBEIT))
master = analyse / "Satzmaster_Agrar_Ernaehrung.csv.gz"
print("Die Wörterbücher werden auf die vollständige Satzbasis angewendet. Die Zuordnung und Prüfung laufen bis zur Ergebnisanzeige.", flush=True)
ausfuehren([PROJEKT / "src/satzmatcher.py",
            "--input", KORPUS / "Saetze_2018-2025.csv.gz",
            "--dictionaries", WOERTERBUCH_ORDNER, "--output", master], zeige_fortschritt=False)
# Die erzeugten Inhalte mit dem festgelegten Analysebestand vergleichen.
# Diese technische Prüfung ist von der inhaltlichen Validierung zu unterscheiden.
pruefung = analyse / "Pruefung_Satzmaster.json"
ausfuehren([PROJEKT / "reproduktion/satzmaster_pruefen.py",
            "--satzmaster", master, "--protokoll", pruefung], zeige_fortschritt=False)
# Das geprüfte Ergebnis an die Excel-Auswertung weitergeben und seine Anzahlen anzeigen.
SATZMASTER, PRUEFUNG_MASTER = master, pruefung
anzahlen = json.loads(pruefung.read_text(encoding="utf-8"))["anzahlen"]
print("A&E-Satzmaster bereit:", f"{anzahlen['agrar_ernaehrung']:,}".replace(",", "."), "Sätze.")
for thema, titel in [("oekologie", "Ökologie"), ("oekonomie", "Ökonomie"), ("sicherheit_resilienz", "Sicherheit und Resilienz")]:
    print(titel + ":", f"{anzahlen[thema]:,}".replace(",", "."), "Sätze.")
print("Die geprüften Inhalte stimmen mit dem bereitgestellten Analysebestand überein.")


## 3. Auswertung und Validierungskennzahlen berechnen

Der Satzmaster wird nach Jahren, Themenbereichen und Fraktionen ausgewertet. Die Gesamtzahlen der Plenarsätze stammen aus der vollständigen Satzbasis aus Schritt 1. Die Arbeitsmappe enthält auch die Monats- und Fraktionswerte der Unterthemen, die Tageszählung zum 20. März 2024 und die Amtsrollenkontrolle. Der Wegweiser ordnet die Ergebnisse den Tabellen und Abbildungen der Arbeit zu.

**Verwendete Dateien:**

- [tabellen_neu_berechnen.py](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/auswertung/tabellen_neu_berechnen.py): berechnet die Auswertung aus den Satzdateien.
- [ooxml_engine.py](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/auswertung/ooxml_engine.py): verarbeitet die Excel-Datei und ihre Formeln.
- [Datengrundlage.xlsx](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/ergebnisse/Datengrundlage.xlsx): enthält den bereitgestellten Ergebnisstand und liefert Tabellenaufbau, Formeln und ergänzende Suchregeln. Das Programm berechnet die Tabellenwerte neu und schreibt eine separate Datei. Die bereitgestellte Arbeitsmappe bleibt unverändert.
- [kennzahlen_berechnen.py](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/validierung/kennzahlen_berechnen.py): berechnet die veröffentlichten Gütemaße und Intervalle aus den vorhandenen [1.000 Handcodierungen](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/validierung/Validierung_AE_und_Themen_1000.xlsx), den [3.000 Handcodierungen der Nichttreffer](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/validierung/Validierung_AE_Recall_3000.xlsx) und den Satzdateien dieses Durchlaufs. Enthalten sind auch die Themenprüfung nach den Vierjahresblöcken 2018–2021 und 2022–2025 sowie der Vergleich der wirtschaftlichen Stichprobenanteile. Die manuellen Codierungen werden dabei unverändert übernommen.

**Neu erzeugt:** `Datengrundlage_neu_berechnet.xlsx`, `Guetemasse_Validierung.csv` und technische Rechenberichte. Die Validierungskennzahlen stehen als [bereitgestellte CSV](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/ergebnisse/Guetemasse_Validierung.csv) ebenfalls im Repository.


In [ ]:
#@title Auswertung und Validierungskennzahlen starten
if SATZMASTER is None or PRUEFUNG_MASTER is None:
    raise RuntimeError("Zuerst Schritt 2 erfolgreich abschließen.")
# Die Vorlage erhalten und für die neu berechneten Ergebnisse einen eigenen Ordner anlegen.
XLSX = PRUEFUNG_EXCEL = GUETEMASSE = PRUEFUNG_VALIDIERUNG = None
DOWNLOADS = {}
auswertung = Path(tempfile.mkdtemp(prefix="auswertung_", dir=ARBEIT))
excel = auswertung / "Datengrundlage_neu_berechnet.xlsx"
pruefung_excel = auswertung / "Pruefung_Excel.json"
guetemasse = auswertung / "Guetemasse_Validierung.csv"
pruefung_validierung = auswertung / "Pruefung_Validierung.json"
print("Die Tabellenwerte werden berechnet und als Excel-Datei gespeichert.", flush=True)
# Satzmaster und vollständige Satzbasis stammen aus demselben Durchlauf.
ausfuehren([PROJEKT / "auswertung/tabellen_neu_berechnen.py",
            "--input", SATZMASTER, "--satzbasis", KORPUS / "Saetze_2018-2025.csv.gz",
            "--dictionaries", WOERTERBUCH_ORDNER,
            "--template", PROJEKT / "ergebnisse/Datengrundlage.xlsx",
            "--output", excel, "--report", pruefung_excel], zeige_fortschritt=False)
print("Die Gütemaße werden aus den vorhandenen Handcodierungen berechnet.", flush=True)
ausfuehren([PROJEKT / "validierung/kennzahlen_berechnen.py",
            "--satzmaster", SATZMASTER,
            "--validierung-themen", PROJEKT / "validierung/Validierung_AE_und_Themen_1000.xlsx",
            "--validierung-recall", PROJEKT / "validierung/Validierung_AE_Recall_3000.xlsx",
            "--satzbasis", KORPUS / "Saetze_2018-2025.csv.gz",
            "--output", guetemasse, "--report", pruefung_validierung], zeige_fortschritt=False)
# Erst nach beiden erfolgreichen Berechnungen das gemeinsame Paket freigeben.
beilage = ablauf.bericht_und_zip_erstellen(
    satzmaster=SATZMASTER, excel=excel, laufbilanz=KORPUS / "Laufbilanz.json",
    manifest=SATZMASTER.with_name(SATZMASTER.name + ".manifest.json"),
    pruefung_satzmaster=PRUEFUNG_MASTER, pruefung_excel=pruefung_excel,
    guetemasse=guetemasse, pruefung_validierung=pruefung_validierung,
    zielordner=auswertung,
    eingabedateien={"Ausgangsdatei": QUELLE,
                   "Handcodierungen A&E/Themen": PROJEKT / "validierung/Validierung_AE_und_Themen_1000.xlsx",
                   "Handcodierungen Nichttreffer": PROJEKT / "validierung/Validierung_AE_Recall_3000.xlsx"},
    herkunft=HERKUNFT)
XLSX, PRUEFUNG_EXCEL = excel, pruefung_excel
GUETEMASSE, PRUEFUNG_VALIDIERUNG = guetemasse, pruefung_validierung
DOWNLOADS = {"A&E-Satzmaster": SATZMASTER, "Excel-Auswertung": XLSX,
             "Validierungskennzahlen": GUETEMASSE, "Ergebnisse als ZIP": beilage["zip"]}
print("Excel-Auswertung, Validierungskennzahlen und gemeinsames Ergebnis-ZIP sind bereit.")


## 4. Ergebnisse herunterladen

Der neu erzeugte Satzmaster, die Excel-Auswertung und die Validierungskennzahlen stehen einzeln oder gemeinsam als ZIP-Datei zum Download bereit. Das Ergebnis-ZIP enthält zusätzlich `Pruefung_Validierung.json` mit den Rechenschritten der Gütemaße und `Bericht_zur_Berechnung.txt` als lesbare Übersicht. Der Gesamtbericht fasst die verwendeten Dateien, Satzzahlen, technischen Prüfungen und die Berechnung aus den vorhandenen Handcodierungen zusammen. Bericht und ZIP werden durch [src/ablauf.py](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/src/ablauf.py) am Ende von Schritt 3 erstellt.

Für einen weiteren Download wird die Auswahl geändert und die Zelle erneut ausgeführt. Die Dateien müssen vor dem Ende der Colab-Sitzung heruntergeladen werden. Die vollständige Satzbasis aus Schritt 1 wird nicht nochmals in das Ergebnis-ZIP aufgenommen.

Der Satzmaster wird als komprimierte CSV-Datei (`.csv.gz`) ausgegeben. Nach dem Entpacken liegt eine CSV-Tabelle mit Semikolon als Trennzeichen vor. Auch die Gütemaße-Datei verwendet Semikolon als Trennzeichen.


In [ ]:
#@title Ergebnis herunterladen
DATEI = "Ergebnisse als ZIP" #@param ["A&E-Satzmaster", "Excel-Auswertung", "Validierungskennzahlen", "Ergebnisse als ZIP"]
# Der Download verwendet die gerade erzeugten Ergebnisse der Schritte 1 bis 3.
if not DOWNLOADS:
    raise RuntimeError("Zuerst die Schritte 1 bis 3 erfolgreich abschließen.")
pfad = DOWNLOADS.get(DATEI)
if pfad is None or not pfad.is_file():
    raise RuntimeError("Die gewählte Ergebnisdatei ist nicht verfügbar.")
from google.colab import files
files.download(str(pfad))


## Ergänzende Unterlagen

- **[Validierung](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/tree/abgabe-v1.0/validierung):** Dokumentation der menschlichen Codierungen für die 1.000er-Prüfung und die 3.000er-Recall-Prüfung. Der Notebook-Lauf berechnet daraus die Gütemaße neu. Die menschlichen Codierentscheidungen werden nicht wiederholt.
- **[Technische Dokumentation](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/docs/Technische_Dokumentation.md):** Erläuterungen zu Aufbereitung, Regeln und Datenfeldern.
- **[Lokale Ausführung](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/docs/Ausfuehrung_lokal.md):** Anleitung zur Berechnung außerhalb von Colab.
- **[Lizenz](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/LICENSE):** Bedingungen für die Nachnutzung.

Die [README](https://github.com/lhaehner/agrarpolitik-bundestag-2018-2025/blob/abgabe-v1.0/README.md) enthält die Übersicht über das Repository. Die technischen Vergleiche prüfen die Reproduktion des festgelegten Analysebestands. Sie ersetzen nicht die inhaltliche Validierung durch menschliche Codierungen.

Quelle: Seán Fobbe, *Corpus der Plenarprotokolle des Deutschen Bundestages*, Version 17.01.2026, [DOI 10.5281/zenodo.18177196](https://doi.org/10.5281/zenodo.18177196).
